<a href="https://colab.research.google.com/github/AndersonGACFilho/AndersonGACFilho/blob/main/notebook_pr%C3%A1tica_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://eramia-rs.sbc.org.br/2026/images/logos/eramia-2026.png" width="60%" alt="ERAMIA - 2026">

# Aprendizado por Reforço: dos Fundamentos à Abstração Temporal

Gabriel de O. Ramos

Arturo de Souza

Augusto A. F. Leal


# Parte 1: MDPs e diferença temporal

Nosso objetivo aqui é entender o processo de decisão de um agente, implementar dois algoritmos tabulares de aprendizado por diferença temporal e visualizar as políticas aprendidas.

**Roteiro:**
- Decision 1D
- Diferença temporal e exploração
- Cliff Walking
- FrozenLake
- Competição no Taxi

## 1. Preparação

Usaremos Gymnasium para os ambientes discretos, NumPy para as tabelas Q, Matplotlib para os gráficos e imageio para os vídeos.

In [1]:
%pip -q install "gymnasium[toy-text]==1.3.0" "imageio>=2.34,<3" "imageio-ffmpeg>=0.5"

In [ ]:
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

import base64
from pathlib import Path

import gymnasium as gym
from gymnasium import Env, spaces
import imageio.v2 as imageio
from IPython.display import display, Video, clear_output, HTML
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from tqdm import tqdm

SEED = 42
print("Gymnasium:", gym.__version__)
print("NumPy:", np.__version__)
print("Tudo pronto... Partiu!")

## 2. Decision 1D: um pequeno MDP

Um processo de decisão de Markov (MDP) é representado por

$$\mathcal{M}=(\mathcal{S},\mathcal{A},P,R,\gamma).$$

Neste exemplo, o agente começa na posição 2, escolhe esquerda ou direita e encontra dois objetivos: um próximo (+3) e outro mais distante (+10). Cada movimento intermediário custa −1.

| Componente | Neste ambiente |
|---|---|
| $\mathcal{S}$ | Posições de 1 a 6 |
| $\mathcal{A}$ | Esquerda ($\leftarrow$) e direita ($\rightarrow$) |
| $P(s'\mid s,a)$ | Movimento determinístico, limitado pelas bordas |
| $R(s,a,s')$ | +3 ao chegar a 1, +10 ao chegar a 6, −1 nos demais movimentos |
| $\gamma$ | Desconto das recompensas futuras |

A propriedade de Markov diz que a próxima transição depende do estado e da ação atuais, não do caminho anterior usado para chegar ali.

As posições 1 e 6 são terminais.

In [ ]:
INITIAL_POSITION = 2
STATES_1D = list(range(1, 7))
ACTIONS_1D = {"←": -1, "→": 1}
TERMINALS_1D = {1: 3, 6: 10}


def step_1d(state, action):
    if state in TERMINALS_1D:
        raise ValueError("Episode already finished.")

    next_state = int(np.clip(state + ACTIONS_1D[action], 1, 6))
    reward = TERMINALS_1D.get(next_state, -1)

    return next_state, reward, next_state in TERMINALS_1D


def run_trajectory_1d(actions):
    state = INITIAL_POSITION
    visited = [state]
    rewards = []

    for action in actions:
        state, reward, done = step_1d(state, action)
        visited.append(state)
        rewards.append(reward)
        if done:
            break

    return visited, rewards


def draw_1d(position):
    fig, ax = plt.subplots(figsize=(10, 1.9))

    for s in STATES_1D:
        color = (
            "#bfdbfe" if s == position
            else "#d1fae5" if s in TERMINALS_1D
            else "#f3f4f6"
        )

        ax.add_patch(plt.Rectangle(
            (s, 0), .94, .95,
            facecolor=color, edgecolor="#475569"
        ))

        ax.text(s + .47, .65, f"s{s}",
                ha="center", va="center", weight="bold")

        label = (
            "Agente" if s == position
            else f"+{TERMINALS_1D[s]}" if s in TERMINALS_1D
            else ""
        )

        ax.text(s + .47, .25, label,
                ha="center", va="center", fontsize=9)

    ax.set_xlim(0.93, 6.99)
    ax.set_ylim(-.06, 1.03)
    ax.axis("off")

    display(fig)
    plt.close(fig)

agent = {
    "position": INITIAL_POSITION,
    "rewards": [],
    "path": [INITIAL_POSITION],
    "done": False
}

output = widgets.Output()

left_btn = widgets.Button(description="← Esquerda")
right_btn = widgets.Button(description="Direita →")
reset_btn = widgets.Button(description="↻ Reiniciar")


def render_agent():
    with output:
        clear_output(wait=True)
        draw_1d(agent["position"])

        print("Trajetória:", " → ".join(map(str, agent["path"])))
        print("Recompensas:", agent["rewards"])
        print("Retorno acumulado:", sum(agent["rewards"]))

        if agent["done"]:
            print("Objetivo alcançado! Episódio encerrado.")

    left_btn.disabled = agent["done"]
    right_btn.disabled = agent["done"]


def move(action):
    if agent["done"]:
        return

    state, reward, done = step_1d(agent["position"], action)

    agent["position"] = state
    agent["rewards"].append(reward)
    agent["path"].append(state)
    agent["done"] = done

    render_agent()


def reset(_):
    agent.update({
        "position": INITIAL_POSITION,
        "rewards": [],
        "path": [INITIAL_POSITION],
        "done": False
    })
    render_agent()


left_btn.on_click(lambda _: move("←"))
right_btn.on_click(lambda _: move("→"))
reset_btn.on_click(reset)

display(widgets.HBox([left_btn, right_btn, reset_btn]))
display(output)

render_agent()

### 2.1 Retorno e fator de desconto

O retorno é a soma das recompensas futuras, descontadas por $\gamma$:

$$G_t=\sum_{k=0}^{T-t-1}\gamma^k R_{t+k+1},\qquad 0\leq\gamma\leq 1.$$

Um $\gamma$ menor favorece recompensas próximas, e valores próximos de 1 tendem a preservar o peso das recompensas futuras.

Vamos comparar duas trajetórias fixas, sem treinamento:

Esquerda: receber +3 imediatamente.
$$G_{\text{esquerda}}=3$$
Direita: receber −1, −1, −1, +10 ao chegar ao objetivo distante.
$$G_{\text{direita}}=-1-\gamma-\gamma^2+10\gamma^3$$

Realize testes alterando os valores de gamma abaixo, na constante GAMMA_1D.

In [ ]:
TRAJECTORIES = {"Close target (←)": ["←"], "Distant target (→)": ["→"] * 4}

def discounted_return(rewards, gamma):
    return sum((gamma ** t) * r for t, r in enumerate(rewards))

GAMMA_1D = 0.8

lines = []
for name, actions in TRAJECTORIES.items():
    visited, rewards = run_trajectory_1d(actions)
    lines.append({"Trajectory": name, "States": " → ".join(map(str, visited)),
                   "Rewards": str(rewards), f"Return (γ={GAMMA_1D})": round(discounted_return(rewards, GAMMA_1D), 3)})
display(pd.DataFrame(lines))

gamma_values = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(8, 3.8))
for name, actions in TRAJECTORIES.items():
    _, rs = run_trajectory_1d(actions)
    ax.plot(gamma_values, [discounted_return(rs, g) for g in gamma_values], label=name)
ax.axvline(GAMMA_1D, color="gray", linestyle="--", alpha=.8, label=f"chosen γ = {GAMMA_1D}")
ax.set(xlabel="Discount factor (γ)", ylabel="Discounted return", title="Discounting can change the optimal decision")
ax.grid(alpha=.25)
ax.legend()
plt.show()

### 2.2 Política e função de valor

Uma política determina como o agente escolhe ações: $\pi(a\mid s)=P(A_t=a\mid S_t=s)$. O valor de ação considera as recompensas futuras esperadas ao tomar $a$ em $s$ e depois seguir $\pi$:

$$Q^\pi(s,a)=\mathbb{E}_\pi[G_t\mid S_t=s,A_t=a].$$

No Decision 1D, os retornos calculados para os dois caminhos ilustram valores de ação para as continuações especificadas. Não iniciamos sabendo esses valores e precisamos aprendê-los, seguindo as regras do exemplo e calculando.

## 3. Aprendizado por diferença temporal (TD)

Em vez de esperar o fim de todas as trajetórias, o agente pode atualizar a estimativa de $Q$ a cada transição, combinando uma recompensa observada e uma estimativa do futuro:

$$Q(s,a)\leftarrow Q(s,a)+\alpha\,\delta,\qquad
\delta=\underbrace{r+\gamma V_{\mathrm{alvo}}}_{\mathrm{alvo\ TD}}-Q(s,a).$$

Aqui, $\alpha$ é a taxa de aprendizado e $\delta$ é o erro TD. SARSA e Q-learning diferem principalmente na definição de $V_{\mathrm{alvo}}$.



### 3.1 Exploração $\varepsilon$-greedy

$$a=\begin{cases}
\text{ação aleatória}, & \text{com probabilidade }\varepsilon,\\
\arg\max_{a'} Q(s,a'), & \text{com probabilidade }1-\varepsilon.
\end{cases}$$

A exploração permite conhecer ações ainda pouco testadas. Durante o treinamento, podemos reduzir $\varepsilon$. Na avaliação final, usamos $\varepsilon=0$ para observar a política gulosa aprendida. Em empates, o código escolhe aleatoriamente entre as ações de maior valor.

### 3.2 SARSA (on-policy)

SARSA usa a próxima ação selecionada pela política comportamental:

$$Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma Q(s',a')-Q(s,a)\right].$$

A atualização considera diretamente a ação $a'$ que o agente escolherá, incluindo o efeito da exploração.



### 3.3 Q-learning (off-policy)

Q-learning usa o maior valor estimado no estado seguinte, independentemente da ação usada para explorar:

$$Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right].$$

Os dois algoritmos aprendem uma tabela $Q(s,a)$. No código abaixo, a diferença entre eles aparece no cálculo do alvo TD.

In [ ]:
def epsilon_greedy_action(Q, state, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    best = np.flatnonzero(np.isclose(Q[state], np.max(Q[state])))
    return int(rng.choice(best))


def linear_epsilon(episode, total, start=1.0, end=0.05, fraction=0.8):
    duration = max(1, int(total * fraction))
    progress = min(1.0, episode / duration)
    return start + (end - start) * progress


def train_td(env_id, algorithm="q_learning", episodes=3000, alpha=0.5, gamma=0.99,
             epsilon_start=1.0, epsilon_end=0.05, decay_fraction=0.8,
             seed=SEED, max_steps=200, env_kwargs=None):
    if algorithm not in ("sarsa", "q_learning"):
        raise ValueError("algorithm must be 'sarsa' or 'q_learning'")
    if not (0 < alpha <= 1 and 0 <= gamma <= 1):
        raise ValueError("Requires 0 < alpha <= 1 and 0 <= gamma <= 1.")
    if not (0 <= epsilon_end <= epsilon_start <= 1):
        raise ValueError("Requires 0 <= epsilon_end <= epsilon_start <= 1.")
    if not (0 < decay_fraction <= 1):
        raise ValueError("decay_fraction must be in (0, 1].")
    if episodes < 1 or max_steps < 1:
        raise ValueError("episodes and max_steps must be positive.")

    env = gym.make(env_id, max_episode_steps=max_steps, **(env_kwargs or {}))
    Q = np.zeros((env.observation_space.n, env.action_space.n), dtype=np.float64)
    rng = np.random.default_rng(seed)
    returns, steps, successes, epsilons = [], [], [], []

    try:
        for ep in tqdm(range(episodes)):
            state, _ = env.reset(seed=seed + ep)
            epsilon = linear_epsilon(ep, episodes, epsilon_start, epsilon_end, decay_fraction)
            action = epsilon_greedy_action(Q, state, epsilon, rng)
            episode_return, last_r, terminated = 0.0, 0.0, False

            for t in range(max_steps):
                next_state, reward, terminated, truncated, _ = env.step(action)
                episode_return += reward
                last_r = reward

                if algorithm == "sarsa":
                    next_action = None if terminated else epsilon_greedy_action(Q, next_state, epsilon, rng)
                    future = 0.0 if terminated else Q[next_state, next_action]
                else:
                    future = 0.0 if terminated else np.max(Q[next_state])

                Q[state, action] += alpha * (reward + gamma * future - Q[state, action])
                state = next_state
                if terminated or truncated:
                    break
                action = next_action if algorithm == "sarsa" else epsilon_greedy_action(Q, state, epsilon, rng)

            returns.append(episode_return)
            steps.append(t + 1)

            success = terminated and (not env_id.startswith("FrozenLake") or last_r > 0)
            successes.append(bool(success))
            epsilons.append(epsilon)
    finally:
        env.close()

    history = {"returns": np.asarray(returns), "steps": np.asarray(steps),
               "successes": np.asarray(successes), "epsilons": np.asarray(epsilons)}
    return Q, history


def moving_average(vector, window=100):
    window = min(window, len(vector))
    return np.convolve(vector, np.ones(window) / window, mode="valid")


def plot_learning(histories, window=100, title="Return Evolution"):
    fig, ax = plt.subplots(figsize=(9, 4))
    for name, h in histories.items():
        y = moving_average(h["returns"], window)
        x = np.arange(len(y)) + min(window, len(h["returns"]))
        ax.plot(x, y, label=name)
    ax.set(xlabel="Training Episode", ylabel=f"Return (moving average of {window})", title=title)
    ax.grid(alpha=.25)
    ax.legend()
    plt.show()

## 4. Cliff Walking: SARSA e Q-learning

O agente deve ir do início **S** ao objetivo **G**, evitando um precipício. Cada passo custa −1. Cair no precipício custa −100 e devolve o agente ao início.

Para mais detalhes sobre esse ambiente, verifique no site do Gymnasium: https://gymnasium.farama.org/environments/toy_text/cliff_walking/

Essa verificação no site do Gymnasium é sempre um passo importante para executar algoritmos de RL em diversos ambientes.

Este é o exemplo clássico para investigar a diferença entre aprender considerando a exploração (SARSA) e aprender em direção à política gulosa (Q-learning). A distinção costuma ser mais visível durante o treinamento, quando $\varepsilon>0$.

Execute os dois algoritmos com os mesmos hiperparâmetros e compare as curvas de retorno.

In [ ]:
CLIFF_ID = "CliffWalking-v1"
CLIFF_EPISODES = 1000
CLIFF_CONFIG = dict(
    episodes=CLIFF_EPISODES,
    alpha=0.1,
    gamma=1.0,
    epsilon_start=0.3,
    epsilon_end=0.1,
    decay_fraction=1.0,
    seed=42,
    max_steps=200,
)

print("Rodando SARSA\t\t", end="")
Q_CLIFF_SARSA, H_CLIFF_SARSA = train_td(CLIFF_ID, algorithm="sarsa", **CLIFF_CONFIG)
print("Rodando Q-learning\t", end="")
Q_CLIFF_Q, H_CLIFF_Q = train_td(CLIFF_ID, algorithm="q_learning", **CLIFF_CONFIG)
plot_learning(
        {"SARSA": H_CLIFF_SARSA,
         "Q-learning": H_CLIFF_Q},
        window=100,
        title=f"Cliff Walking learning curves (ε = {CLIFF_CONFIG["epsilon_start"]})")

### 4.1 Avaliação de resultados

Após o treinamento, executaremos novos episódios sem exploração ($\varepsilon=0$). As condições iniciais usam sementes predefinidas, iguais para os dois métodos.

- Retorno médio: soma de recompensas por episódio (maior é melhor).
- Taxa de sucesso: proporção de episódios concluídos no objetivo.
- Passos médios: comprimento dos episódios (menor é melhor, se o sucesso for comparável).

É importante considerarmos que os resultados da avaliação gulosa não se equivalem aos retornos obtidos com exploração durante o treinamento.

In [ ]:
def evaluate_policy(Q, env_id, episodes=100, evaluation_seed=20_000,
                    max_steps=200, env_kwargs=None):

    env = gym.make(env_id, max_episode_steps=max_steps, **(env_kwargs or {}))
    results = []
    try:
        for ep in range(episodes):
            seed = evaluation_seed + ep
            state, _ = env.reset(seed=seed)
            rng = np.random.default_rng(seed + 1_000_000)
            episode_return, terminated, last_r = 0.0, False, 0.0
            for t in range(max_steps):
                action = epsilon_greedy_action(Q, state, 0.0, rng)
                state, reward, terminated, truncated, _ = env.step(action)
                episode_return += reward
                last_r = reward
                if terminated or truncated:
                    break
            success = terminated and (not env_id.startswith("FrozenLake") or last_r > 0)
            results.append({"seed": seed, "return": episode_return, "steps": t + 1,
                             "success": bool(success)})
    finally:
        env.close()
    return pd.DataFrame(results)


def summarize_evaluation(evaluation):
    return {"Mean return": round(evaluation["return"].mean(), 2),
            "Success (%)": round(100 * evaluation["success"].mean(), 1),
            "Mean steps": round(evaluation["steps"].mean(), 1)}

EVALUATION_CLIFF = {
    "SARSA": evaluate_policy(Q_CLIFF_SARSA, CLIFF_ID, episodes=100, evaluation_seed=20_000),
    "Q-learning": evaluate_policy(Q_CLIFF_Q, CLIFF_ID, episodes=100, evaluation_seed=20_000),
}
display(pd.DataFrame({name: summarize_evaluation(df) for name, df in EVALUATION_CLIFF.items()}).T)

### 4.2 Política aprendida

As setas mostram a ação gulosa em cada posição. As células da borda inferior contém o precipício, exceto o o local início e o objetivo. Abaixo da tabela de setas, podemos assistir ao agente executando um episódio.

In [ ]:
def draw_cliff_policy(Q, title):
    arrows = {0: "↑", 1: "→", 2: "↓", 3: "←"}
    fig, ax = plt.subplots(figsize=(11, 3.3))
    for row in range(4):
        for col in range(12):
            s = row * 12 + col
            cliff = row == 3 and 1 <= col <= 10
            start, goal = s == 36, s == 47
            color = "#fecaca" if cliff else "#d1fae5" if goal else "#dbeafe" if start else "#f3f4f6"
            ax.add_patch(plt.Rectangle((col, 3-row), 1, 1, edgecolor="#94a3b8", facecolor=color))
            label = "S" if start else "G" if goal else "X" if cliff else arrows[int(np.argmax(Q[s]))]
            ax.text(col + 0.5, 3.5-row, label, ha="center", va="center", fontsize=13)
    ax.set(xlim=(0, 12), ylim=(0, 4), title=title)
    ax.axis("off")
    plt.show()

draw_cliff_policy(Q_CLIFF_SARSA, "SARSA (greedy policy after training)")
draw_cliff_policy(Q_CLIFF_Q, "Q-learning (greedy policy after training)")

In [ ]:
def show_policy_video(Q, env_id, filename="episode.mp4", seed=20_000,
                      max_steps=100, fps=5, env_kwargs=None):
    import os
    filename = str(filename)
    env = gym.make(env_id, render_mode="rgb_array", max_episode_steps=max_steps,
                   **(env_kwargs or {}))
    rng = np.random.default_rng(seed + 1_000_000)
    frames = []
    episode_return, terminated, truncated, steps, last_r = 0.0, False, False, 0, 0.0
    try:
        state, _ = env.reset(seed=seed)
        frames.append(np.asarray(env.render()).copy())
        for steps in range(1, max_steps + 1):
            action = epsilon_greedy_action(Q, state, 0.0, rng)
            state, reward, terminated, truncated, _ = env.step(action)
            last_r = reward
            episode_return += reward
            frames.append(np.asarray(env.render()).copy())
            if terminated or truncated:
                break
    finally:
        env.close()

    def pad_to_macroblock(frame, block_size=16):
        h, w = frame.shape[:2]
        pad_h = (block_size - (h % block_size)) % block_size
        pad_w = (block_size - (w % block_size)) % block_size
        return np.pad(frame, ((0, pad_h), (0, pad_w), (0, 0)), mode="edge")

    with imageio.get_writer(filename, format="FFMPEG", mode="I", fps=fps, codec="libx264") as writer:
        for frame in frames:
            writer.append_data(pad_to_macroblock(frame))

    from IPython.display import HTML
    import base64
    with open(filename, "rb") as f:
        video_bytes = f.read()
    data_url = "data:video/mp4;base64," + base64.b64encode(video_bytes).decode()
    display(HTML(f"""
    <video width="450" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))

    success = terminated and (not env_id.startswith("FrozenLake") or last_r > 0)
    print(f"Return: {episode_return:.1f} | Displayed steps: {steps} | Success: {'yes' if success else 'no'}")
    if truncated and not terminated:
        print("The visualization ended due to the step limit, without reaching the goal.")

In [ ]:
print("Cliff Walking demonstration with SARSA:")
show_policy_video(Q_CLIFF_SARSA, CLIFF_ID, filename="cliff_sarsa.mp4", seed=20_000, max_steps=100, fps=5)

print("\nCliff Walking demonstration with Q-learning:")
show_policy_video(Q_CLIFF_Q, CLIFF_ID, filename="cliff_qlearning.mp4", seed=20_000, max_steps=100, fps=5)

## 5. FrozenLake: recompensas esparsas e estocasticidade

No FrozenLake, o agente começa no estado inicial S, e deve alcançar o goal G, evitando os buracos H. Ele recebe +1 ao atingir o objetivo e 0 nos demais passos.

Com is_slippery=True, a ação escolhida não determina o próximo movimento com exatidão. Por conta disso, o FrozenLake proporciona duas dificuldades: exploração com recompensa esparsa e transições estocásticas.

In [ ]:
FROZEN_ID = "FrozenLake-v1"
FROZEN_KWARGS = {"map_name": "4x4", "is_slippery": True}

FROZEN_CONFIG = dict(episodes=15_000, alpha=0.3, gamma=0.99,
                     epsilon_start=1.0, epsilon_end=0.05, decay_fraction=0.85,
                     seed=123, max_steps=100, env_kwargs=FROZEN_KWARGS)

Q_FROZEN_SARSA, H_FROZEN_SARSA = train_td(FROZEN_ID, algorithm="sarsa", **FROZEN_CONFIG)
Q_FROZEN_Q, H_FROZEN_Q = train_td(FROZEN_ID, algorithm="q_learning", **FROZEN_CONFIG)

plot_learning({"SARSA": H_FROZEN_SARSA, "Q-learning": H_FROZEN_Q}, window=500,
              title="Slippery FrozenLake 4×4 (Training returns)")

EVALUATION_FROZEN = {
    "SARSA": evaluate_policy(Q_FROZEN_SARSA, FROZEN_ID, episodes=300, evaluation_seed=30_000,
                             max_steps=100, env_kwargs=FROZEN_KWARGS),
    "Q-learning": evaluate_policy(Q_FROZEN_Q, FROZEN_ID, episodes=300, evaluation_seed=30_000,
                                  max_steps=100, env_kwargs=FROZEN_KWARGS),
}
display(pd.DataFrame({name: summarize_evaluation(df) for name, df in EVALUATION_FROZEN.items()}).T)

### 5.1 Política e vídeo

Mesmo com uma boa política, uma ação pode levar a um resultado inesperado no gelo. Por isso, um único vídeo não necessariamente irá medir a qualidade da política.

Vale a pena aqui comparar is_slippery=True e False, reexecutando o treinamento e avaliando com a mesma configuração. Tente também alterar $\varepsilon$ e $\gamma$.

In [ ]:
def draw_frozen_policy(Q, env_kwargs=None, title="FrozenLake"):
    env = gym.make(FROZEN_ID, **(env_kwargs or {}))
    try:
        desc = env.unwrapped.desc
        grid = tuple("".join(c.decode("utf-8") for c in row) for row in desc)
    finally:
        env.close()
    arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}
    n = len(grid)
    fig, ax = plt.subplots(figsize=(5, 5))
    for r, row in enumerate(grid):
        for c, cell_type in enumerate(row):
            color = {"S": "#dbeafe", "F": "#f3f4f6", "H": "#fecaca", "G": "#d1fae5"}[cell_type]
            ax.add_patch(plt.Rectangle((c, n-1-r), 1, 1, facecolor=color, edgecolor="#94a3b8"))
            label = cell_type if cell_type in "SHG" else arrows[int(np.argmax(Q[r*n+c]))]
            ax.text(c+.5, n-r-.5, label, ha="center", va="center", fontsize=16)
    ax.set(xlim=(0, n), ylim=(0, n), title=title)
    ax.axis("off")
    plt.show()

draw_frozen_policy(Q_FROZEN_SARSA, env_kwargs=FROZEN_KWARGS, title="FrozenLake (SARSA)")
draw_frozen_policy(Q_FROZEN_Q, env_kwargs=FROZEN_KWARGS, title="FrozenLake (Q-learning)")

In [ ]:
print("FrozenLake demonstration with SARSA:")
show_policy_video(Q_FROZEN_SARSA, FROZEN_ID, filename="frozen_sarsa.mp4", seed=30_000,
                  max_steps=70, fps=5, env_kwargs=FROZEN_KWARGS)

print("FrozenLake demonstration with Q-learning:")
show_policy_video(Q_FROZEN_Q, FROZEN_ID, filename="frozen_qlearning.mp4", seed=30_000,
                  max_steps=70, fps=5, env_kwargs=FROZEN_KWARGS)

## 6. Competição no ambiente Taxi

![Taxi Driver](https://www.tenhomaisdiscosqueamigos.com/wp-content/uploads/2016/03/taxi-driver-robert-de-niro.jpg)

No Taxi, o agente deve buscar um passageiro, transportá-lo e desembarcá-lo no destino. Há 500 estados, 6 ações (quatro deslocamentos, embarcar e desembarcar). As recompensas são: −1 por passo, −10 por operação inválida e +20 por entrega correta.

Taxi: https://gymnasium.farama.org/environments/toy_text/taxi/

### 6.1 Regras

1. Cada participante escolhe SARSA ou Q-learning e altera $\alpha$, $\gamma$, $\varepsilon_{\mathrm{início}}$, $\varepsilon_{\mathrm{fim}}$ e a fração de decaimento.
2. Cada um treina por 5.000 episódios, com no máximo 200 passos por episódio, usando o Taxi padrão (sem chuva e sem passageiro instável).
3. Não é permitido realizar alterações no ambiente, modificar seeds ou o protocolo de treinamento definido.
4. Avaliação gulosa em 200 episódios com as mesmas sementes de teste para todos.
5. Classificação pelo maior retorno médio. Desempate: (a) maior taxa de sucesso e, depois, (b) menos passos médios.

O objetivo de cada participante é ajustar os hiperparâmetros para obter uma política eficiente e confiável.

In [ ]:
TAXI_ID = "Taxi-v4"
TAXI_EPISODES = 5_000
TAXI_STEPS = 200
TAXI_TRAIN_SEED = 2026
TAXI_TEST_SEED = 90_000

TAXI_ENV_KWARGS = None
#TAXI_ENV_KWARGS = {"is_rainy": True, "rainy_probability": 0.8, "fickle_passenger": True, "fickle_probability": 0.3}

BASELINE = "Baseline (SARSA)"
MY_NAME = "Taxi do De Niro"

TAXI_CONFIGURATIONS = {
    BASELINE: dict(
        algorithm="sarsa", alpha=0.45, gamma=0.8,
        epsilon_start=1.0, epsilon_end=0.01, decay_fraction=0.8,
    ),
    MY_NAME: dict(
        algorithm="sarsa", alpha=0.45, gamma=0.8,
        epsilon_start=0.9, epsilon_end=0.01, decay_fraction=0.8,
    ),
}

Q_TAXI = {}
H_TAXI = {}
TAXI_EVALUATIONS = {}
result_rows = []

print("Configuração de Ambiente do Taxi ===")
print(f"ID do Ambiente: {TAXI_ID}")
print(f"Parâmetros extras (env_kwargs): {TAXI_ENV_KWARGS}\n")

for name, config in TAXI_CONFIGURATIONS.items():
    print(f"Training: {name} ({config['algorithm']})")
    Q, history = train_td(
        TAXI_ID, episodes=TAXI_EPISODES, seed=TAXI_TRAIN_SEED,
        max_steps=TAXI_STEPS, env_kwargs=TAXI_ENV_KWARGS, **config,
    )
    evaluation = evaluate_policy(
        Q, TAXI_ID, episodes=200, evaluation_seed=TAXI_TEST_SEED,
        max_steps=TAXI_STEPS, env_kwargs=TAXI_ENV_KWARGS,
    )
    Q_TAXI[name] = Q
    H_TAXI[name] = history
    TAXI_EVALUATIONS[name] = evaluation
    result_rows.append({"Participant": name, "Algorithm": config["algorithm"],
                         **summarize_evaluation(evaluation)})

plot_learning({BASELINE: H_TAXI[BASELINE], MY_NAME: H_TAXI[MY_NAME]}, window=100,
              title="Taxi")

TAXI_RANKING = pd.DataFrame(result_rows).sort_values(
    ["Mean return", "Success (%)", "Mean steps"], ascending=[False, False, True]
).reset_index(drop=True)
TAXI_RANKING.index = TAXI_RANKING.index + 1
TAXI_RANKING.index.name = "Position"
display(TAXI_RANKING)

### 6.2 Vídeo do resultado

Alterar $\gamma$ modifica o peso da entrega futura e alterar $\varepsilon$ muda o quanto o agente explora durante o treinamento.

In [ ]:
show_policy_video(Q_TAXI[MY_NAME], TAXI_ID, filename="taxi_competicao.mp4",
                  seed=TAXI_TEST_SEED, max_steps=100, fps=8)

# Parte 2: Aprendizado por reforço hierárquico

E se no Taxi tratássemos a tarefa como duas habilidades temporalmente estendidas? Por exemplo, ir até o passageiro e ir até o destino. As políticas aprendidas por SARSA e Q-learning escolheriam ações primitivas a cada passo, e um macanismo superior definiria uma política por um período de tempo para executar essas ações.

Nesta próxima parte do minicurso, introduziremos o conceito options, que encapsulam comportamentos que duram vários passos. Essa é a motivação para passar de decisões de um único passo a abstração temporal e a processos de decisão semi-Markov (SMDPs).

**Roteiro:**
- Four Rooms
- Option-critic
- Análise dos resultados

## 7. Options com option-critic

O algoritmo option-critic aprende simultaneamente quais options escolher, como cada options age e em que estados ela termina.

Cada ação primitiva é um movimento: cima, baixo, esquerda ou direita. O agente começa em uma posição livre escolhida ao acaso. Ao alcançar o objetivo, recebe recompensa +1 e o episódio termina. Para representar incerteza, após um movimento válido há probabilidade 1/3 de o agente ser deslocado para uma célula vizinha livre.

### 7.1 Ambiente Fourrooms


In [ ]:
# @title
class Fourrooms(Env):
    def __init__(self, seed=0):
        layout = """\
wwwwwwwwwwwww
w     w     w
w     w     w
w           w
w     w     w
w     w     w
ww wwww     w
w     www www
w     w     w
w     w     w
w           w
w     w     w
wwwwwwwwwwwww
"""
        self.occupancy = np.array([
            [1 if cell == "w" else 0 for cell in line]
            for line in layout.splitlines()
        ])
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Discrete(int(np.sum(self.occupancy == 0)))
        self.directions = [
            np.array((-1, 0)),  # cima
            np.array((1, 0)),   # baixo
            np.array((0, -1)),  # esquerda
            np.array((0, 1)),   # direita
        ]

        self.tostate = {}
        for i in range(13):
            for j in range(13):
                if self.occupancy[i, j] == 0:
                    self.tostate[(i, j)] = len(self.tostate)
        self.tocell = {state: cell for cell, state in self.tostate.items()}

        self.goal = 62
        self.init_states = list(range(self.observation_space.n))
        self.init_states.remove(self.goal)
        self.possible_next_goals = [
            68, 69, 70, 71, 72, 78, 79, 80, 81, 82, 88, 89, 90,
            91, 92, 93, 99, 100, 101, 102, 103,
        ]
        self.rng = np.random.RandomState(seed)

    def switch_goal(self):
        self.goal = self.rng.choice(self.possible_next_goals)

    def empty_around(self, cell):
        available = []
        for action in range(self.action_space.n):
            next_cell = tuple(cell + self.directions[action])
            if not self.occupancy[next_cell]:
                available.append(next_cell)
        return available

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.RandomState(seed)
        state = self.rng.choice(self.init_states)
        self.currentcell = self.tocell[state]
        return state, {}

    def step(self, action):
        next_cell = tuple(self.currentcell + self.directions[action])
        if not self.occupancy[next_cell]:
            self.currentcell = next_cell
            if self.rng.uniform() < 1.0 / 3:
                neighbors = self.empty_around(self.currentcell)
                self.currentcell = neighbors[self.rng.randint(len(neighbors))]

        state = self.tostate[self.currentcell]
        terminated = state == self.goal
        return state, float(terminated), terminated, False, {}


### 7.2 Funções de visualização

In [ ]:
def logsumexp(x):
    maximum = x.max()
    return maximum + np.log(np.sum(np.exp(x - maximum)))


def _draw_grid(ax, occupancy):
    for i in range(occupancy.shape[0]):
        for j in range(occupancy.shape[1]):
            if occupancy[i, j] == 1:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1,
                                       color="gray", ec="none"))


def _mark_cell(ax, env, state, color):
    i, j = env.tocell[state]
    ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, color=color,
                           alpha=0.6, ec="none", zorder=10))


def plot_rewards_lengths(episode_rewards, episode_lengths, save_path):
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()
    window = min(50, len(episode_rewards) // 2 or 1)
    ax1.plot(episode_rewards, alpha=0.3, color="steelblue", linewidth=0.5)
    ax2.plot(episode_lengths, alpha=0.3, color="coral", linewidth=0.5)

    if window > 1:
        kernel = np.ones(window) / window
        smoothed_rewards = np.convolve(episode_rewards, kernel, mode="valid")
        smoothed_lengths = np.convolve(episode_lengths, kernel, mode="valid")
        x = np.arange(len(smoothed_rewards)) + window // 2
        ax1.plot(x, smoothed_rewards, color="steelblue", linewidth=2,
                 label=f"Retorno médio (janela={window})")
        ax2.plot(x, smoothed_lengths, color="coral", linewidth=2,
                 label=f"Passos médios (janela={window})")

    ax1.set_xlabel("Episódio")
    ax1.set_ylabel("Retorno", color="steelblue")
    ax2.set_ylabel("Duração do episódio", color="coral")
    ax1.set_title("Retorno e duração ao longo do treinamento")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    if lines1 or lines2:
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
    fig.tight_layout()
    fig.savefig(save_path)
    return fig


def plot_policy_quivers(env, option_policies, save_path):
    noptions = len(option_policies)
    fig, axes = plt.subplots(1, noptions, figsize=(5 * noptions, 5))
    if noptions == 1:
        axes = [axes]
    directions = [np.array((-1, 0)), np.array((1, 0)),
                  np.array((0, -1)), np.array((0, 1))]

    for option_index, policy in enumerate(option_policies):
        ax = axes[option_index]
        _draw_grid(ax, env.occupancy)
        _mark_cell(ax, env, env.goal, "green")
        X, Y, U, V = [], [], [], []
        for state in range(env.observation_space.n):
            i, j = env.tocell[state]
            probabilities = policy.pmf(np.array([state]))
            di = probabilities @ np.array([d[0] for d in directions])
            dj = probabilities @ np.array([d[1] for d in directions])
            X.append(j); Y.append(i); U.append(dj); V.append(di)
        ax.quiver(X, Y, U, V, angles="xy", scale_units="xy", scale=1.5,
                  color="black", width=0.006)
        ax.set_title(f"Opção {option_index}")
        ax.set_xlim(-0.5, 12.5); ax.set_ylim(12.5, -0.5)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    fig.savefig(save_path)
    return fig


def plot_termination_heatmaps(env, terminations, save_path):
    noptions = len(terminations)
    fig, axes = plt.subplots(1, noptions, figsize=(5 * noptions, 5))
    if noptions == 1:
        axes = [axes]
    grid = np.full((13, 13), np.nan)
    for option_index, termination in enumerate(terminations):
        ax = axes[option_index]
        grid.fill(np.nan)
        for state in range(env.observation_space.n):
            i, j = env.tocell[state]
            grid[i, j] = termination.pmf(np.array([state]))
        image = ax.imshow(grid, cmap="Reds", vmin=0, vmax=1, origin="upper")
        _draw_grid(ax, env.occupancy)
        _mark_cell(ax, env, env.goal, "green")
        ax.set_title(f"Opção {option_index}")
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(save_path)
    return fig


def plot_option_initiation_heatmaps(env, policy, save_path):
    noptions = policy.weights.shape[1]
    fig, axes = plt.subplots(1, noptions, figsize=(5 * noptions, 5))
    if noptions == 1:
        axes = [axes]
    grid = np.full((13, 13), np.nan)
    for option_index in range(noptions):
        ax = axes[option_index]
        grid.fill(np.nan)
        for state in range(env.observation_space.n):
            i, j = env.tocell[state]
            grid[i, j] = policy.pmf(np.array([state]))[option_index]
        image = ax.imshow(grid, cmap="Blues", vmin=0, vmax=1, origin="upper")
        _draw_grid(ax, env.occupancy)
        _mark_cell(ax, env, env.goal, "green")
        ax.set_title(f"Opção {option_index}")
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    fig.savefig(save_path)
    return fig


### 7.3 Componentes do option-critic

O Option-Critic aprende quais opções escolher, quais ações executar e quando terminar uma opção.

- **Política de alto nível:** $\pi_\Omega(\omega\mid s)$, escolhe uma opção por *softmax*.
- **Política interna:** $\pi_\omega(a\mid s)$, escolhe ações dentro da opção por *softmax*.
- **Função de término:** $\beta_\omega(s)$, representa a probabilidade de encerrar uma opção, calculada por uma sigmoide.
- **Crítico:** $Q_\Omega(s,\omega)$, estima o valor de executar uma opção em determinado estado.

Atualização do crítico, semelhante ao Q-learning:

Dado que:

$$
\delta=r+\gamma U(s',\omega)-Q_\Omega(s,\omega)
$$

Então

$$
Q_\Omega(s,\omega)\leftarrow Q_\Omega(s,\omega)+\alpha\delta
$$

O valor futuro $U(s',\omega)$ combina duas possibilidades:

$$
U(s',\omega)=
(1-\beta_\omega(s'))Q_\Omega(s',\omega)
+\beta_\omega(s')\max_{\omega'}Q_\Omega(s',\omega')
$$

- Continuar: com probabilidade $1-\beta_\omega(s')$, mantém o valor da opção atual.
- Terminar: com probabilidade $\beta_\omega(s')$, considera o valor da melhor opção disponível.

Por exemplo, se $\beta_\omega(s')=0{,}2$, existe 80% de probabilidade de continuar a opção e 20% de encerrá-la. A equação calcula o valor esperado dessas possibilidades.

Atualização do crítico de ação: além de $Q_\Omega$, o agente aprende $Q_U(s,\omega,a)$, que estima o valor de executar uma ação dentro de uma opção. Sua atualização também utiliza o erro TD:

$$
\delta_U=r+\gamma U(s',\omega)-Q_U(s,\omega,a)
$$

$$
Q_U(s,\omega,a)\leftarrow Q_U(s,\omega,a)+\alpha\delta_U
$$

Atualização da política interna: o crítico de ação $Q_U(s,\omega,a)$ fornece o sinal para aumentar ou diminuir a probabilidade das ações:

$$
\theta_\omega\leftarrow\theta_\omega+
\alpha_\pi Q_U(s,\omega,a)
\nabla_{\theta_\omega}\log\pi_\omega(a\mid s)
$$

Atualização da função de término: utiliza a vantagem da opção atual em relação à melhor opção disponível:

$$
A_\Omega(s,\omega)=Q_\Omega(s,\omega)-\max_{\omega'}Q_\Omega(s,\omega')
$$

$$
\vartheta_\omega\leftarrow\vartheta_\omega-
\alpha_\beta A_\Omega(s,\omega)
\nabla_{\vartheta_\omega}\beta_\omega(s)
$$

- Se outra opção possui valor maior, o aprendizado aumenta a probabilidade de terminar a opção atual.
- Se a opção atual já possui o maior valor, a vantagem é zero e essa atualização não modifica o término.

Assim, o agente aprende simultaneamente o valor das opções, suas políticas de ações e quando deve encerrá-las.

In [ ]:
class SoftmaxPolicy:
    def __init__(self, rng, nfeatures, nactions, temp=1.0):
        self.rng = rng
        self.weights = np.zeros((nfeatures, nactions))
        self.temp = temp

    def value(self, phi, action=None):
        if action is None:
            return np.sum(self.weights[phi, :], axis=0)
        return np.sum(self.weights[phi, action], axis=0)

    def pmf(self, phi):
        values = self.value(phi) / self.temp
        return np.exp(values - logsumexp(values))

    def sample(self, phi):
        return int(self.rng.choice(self.weights.shape[1], p=self.pmf(phi)))


class SigmoidTermination:
    def __init__(self, rng, nfeatures):
        self.rng = rng
        self.weights = np.zeros((nfeatures,))

    def pmf(self, phi):
        z = np.sum(self.weights[phi])
        return 1.0 / (1.0 + np.exp(-z))

    def sample(self, phi):
        return int(self.rng.uniform() < self.pmf(phi))

    def grad(self, phi):
        probability = self.pmf(phi)
        return probability * (1.0 - probability), phi


class IntraOptionQLearning:
    def __init__(self, discount, lr, reg, terminations, weights):
        self.lr = lr
        self.discount = discount
        self.reg = reg
        self.terminations = terminations
        self.weights = weights

    def start(self, phi, option):
        self.last_phi = phi
        self.last_option = option
        self.last_value = self.value(phi, option)

    def value(self, phi, option=None):
        if option is None:
            return np.sum(self.weights[phi, :], axis=0)
        return np.sum(self.weights[phi, option], axis=0)

    def advantage(self, phi, option=None):
        values = self.value(phi)
        advantages = values - np.max(values) + self.reg
        if option is None:
            return advantages
        return advantages[option]

    def update(self, phi, option, reward, done):
        target = reward
        if not done:
            current_values = self.value(phi)
            termination = self.terminations[self.last_option].pmf(phi)
            target += self.discount * (
                (1.0 - termination) * current_values[self.last_option]
                + termination * np.max(current_values)
            )
        td_error = target - self.last_value
        self.weights[self.last_phi, self.last_option] += self.lr * td_error
        if not done:
            self.last_value = current_values[option]
        self.last_option = option
        self.last_phi = phi
        return target


class IntraOptionActionQLearning:
    def __init__(self, discount, lr, terminations, weights, critic):
        self.lr = lr
        self.discount = discount
        self.terminations = terminations
        self.weights = weights
        self.critic = critic

    def value(self, phi, option, action):
        return np.sum(self.weights[phi, option, action], axis=0)

    def start(self, phi, option, action):
        self.last_phi = phi
        self.last_option = option
        self.last_action = action

    def update(self, phi, option, action, reward, done):
        target = reward
        if not done:
            current_values = self.critic.value(phi)
            termination = self.terminations[self.last_option].pmf(phi)
            target += self.discount * (
                (1.0 - termination) * current_values[self.last_option]
                + termination * np.max(current_values)
            )
        td_error = target - self.value(
            self.last_phi, self.last_option, self.last_action
        )
        self.weights[self.last_phi, self.last_option, self.last_action] += (
            self.lr * td_error
        )
        self.last_phi = phi
        self.last_option = option
        self.last_action = action


class TerminationGradient:
    def __init__(self, terminations, critic, lr):
        self.terminations = terminations
        self.critic = critic
        self.lr = lr

    def update(self, phi, option):
        magnitude, direction = self.terminations[option].grad(phi)
        self.terminations[option].weights[direction] -= (
            self.lr * magnitude * self.critic.advantage(phi, option)
        )


class IntraOptionGradient:
    def __init__(self, option_policies, lr):
        self.lr = lr
        self.option_policies = option_policies

    def update(self, phi, option, action, critic_feedback):
        probabilities = self.option_policies[option].pmf(phi)
        self.option_policies[option].weights[phi, :] -= (
            self.lr * critic_feedback * probabilities
        )
        self.option_policies[option].weights[phi, action] += (
            self.lr * critic_feedback
        )


### 7.4 Treinamento

O loop atualiza o crítico, a política interna da opção e sua função de término a cada transição.

In [ ]:
def train_options(
    discount=0.99,
    lr_critic=0.5,
    lr_intra=0.25,
    lr_term=0.25,
    temperature=0.001,
    nepisodes=2000,
    nsteps=1000,
    noptions=4,
    baseline=False,
    seed=42,
    reg=0.0,
    log_every=200,
):
    env = Fourrooms(seed=seed)
    nfeatures = env.observation_space.n
    nactions = env.action_space.n
    rng = np.random.RandomState(seed)

    option_policies = [
        SoftmaxPolicy(rng, nfeatures, nactions, temperature)
        for _ in range(noptions)
    ]
    option_terminations = [
        SigmoidTermination(rng, nfeatures) for _ in range(noptions)
    ]
    policy = SoftmaxPolicy(rng, nfeatures, noptions, temperature)
    critic = IntraOptionQLearning(
        discount, lr_critic, reg, option_terminations, policy.weights
    )
    action_weights = np.zeros((nfeatures, noptions, nactions))
    action_critic = IntraOptionActionQLearning(
        discount, lr_critic, option_terminations, action_weights, critic
    )
    termination_improvement = TerminationGradient(
        option_terminations, critic, lr_term
    )
    intra_improvement = IntraOptionGradient(option_policies, lr_intra)

    episode_rewards, episode_lengths, option_switch_counts = [], [], []
    for episode in range(nepisodes):
        phi, _ = env.reset()
        phi = np.array([phi])
        option = policy.sample(phi)
        action = option_policies[option].sample(phi)
        critic.start(phi, option)
        action_critic.start(phi, option, action)

        cumulative_reward = 0.0
        switches = 0
        average_duration = 0.0
        duration = 1
        for step in range(nsteps):
            observation, reward, terminated, truncated, _ = env.step(action)
            phi = np.array([observation])

            previous_option = option
            if not (terminated or truncated) and option_terminations[option].sample(phi):
                option = policy.sample(phi)
                switches += 1
                average_duration += (1.0 / switches) * (duration - average_duration)
                duration = 1

            action = option_policies[option].sample(phi)
            critic.update(phi, option, reward, terminated)
            action_critic.update(phi, option, action, reward, terminated)

            if not (terminated or truncated):
                feedback = action_critic.value(phi, option, action)
                if baseline:
                    feedback -= critic.value(phi, option)
                intra_improvement.update(phi, option, action, feedback)
                termination_improvement.update(phi, previous_option)

            cumulative_reward += reward
            duration += 1
            if terminated or truncated:
                break

        episode_rewards.append(cumulative_reward)
        episode_lengths.append(step + 1)
        option_switch_counts.append(switches)
        if log_every and ((episode + 1) % log_every == 0 or episode == 0):
            print(
                f"episódio={episode + 1:4d}  passos={step + 1:3d}  "
                f"retorno={cumulative_reward:1.1f}  trocas={switches:3d}  "
                f"duração média da opção={average_duration:.1f}"
            )

    return (env, policy, option_policies, option_terminations,
            episode_rewards, episode_lengths, option_switch_counts)


Hiperparâmetros:

| Hiperparâmetro | Significado |
|---|---|
| discount ($\gamma$) | Fator de desconto para valorizar recompensas futuras (próximo a 1.0 prioriza o longo prazo). |
| lr_critic ($\alpha$) | Taxa de aprendizado do crítico (Intra-Option Q-learning) para atualizar os valores de estado-opção. |
| lr_intra | Taxa de aprendizado para a política interna de cada opção (Intra-Option Gradient). |
| lr_term | Taxa de aprendizado para a função de término de cada opção (Termination Gradient). |
| temperature ($\tau$) | Temperatura do Softmax que controla a exploração (valores baixos tornam a seleção mais gulosa). |
| nepisodes | Número total de episódios para o treinamento. |
| nsteps | Limite máximo de passos permitidos por episódio. |
| noptions | Quantidade de opções abstratas (habilidades) a serem aprendidas simultaneamente. |
| baseline | Se True, usa o valor do crítico como linha de base para reduzir a variância do gradiente da política. |
| reg | Termo de regularização para encorajar ou penalizar a troca constante de opções. |

In [ ]:
OPTIONS_CONFIG = dict(
    discount=0.99,
    lr_critic=0.5,
    lr_intra=0.25,
    lr_term=0.25,
    temperature=0.001,
    nepisodes=2000,
    nsteps=1000,
    noptions=4,
    baseline=False,
    seed=42,
    reg=0.0,
    log_every=200,
)

(env_options, policy_options, option_policies, option_terminations,
 rewards_options, lengths_options, switches_options) = train_options(**OPTIONS_CONFIG)

print(f"\nEstados: {env_options.observation_space.n}")
print(f"Ações primitivas: {env_options.action_space.n}")
print(f"Opções aprendidas: {len(option_policies)}")


### 7.5 Leitura dos resultados

O primeiro gráfico acompanha o retorno e o número de passos por episódio. As setas mostram a direção média das ações, ponderada pelas probabilidades da política interna de cada opção. Os mapas de iniciação mostram a probabilidade de escolher cada opção ao começar em um estado; os mapas de término mostram onde cada opção aprendeu maior probabilidade de encerrar.

As opções não recebem nomes como “ir para a sala norte”: esses comportamentos emergem dos parâmetros aprendidos. Compare os mapas para investigar se opções diferentes desenvolveram estratégias distintas.


In [ ]:
reward_fig = plot_rewards_lengths(
    rewards_options, lengths_options, "fourrooms_retorno_duracao.pdf"
)
#reward_fig

In [ ]:
policy_fig = plot_policy_quivers(
    env_options, option_policies, "fourrooms_politicas_opcoes.pdf"
)
#policy_fig

In [ ]:
init_fig = plot_option_initiation_heatmaps(
    env_options, policy_options, "fourrooms_iniciacao_opcoes.pdf"
)
#init_fig

In [ ]:
term_fig = plot_termination_heatmaps(
    env_options, option_terminations, "fourrooms_terminacao_opcoes.pdf"
)
#term_fig

# Referências

- BACON, Pierre-Luc; HARB, Jean; PRECUP, Doina. **The Option-Critic Architecture**. In: AAAI Conference on Artificial Intelligence, 31., 2017. DOI: 10.1609/aaai.v31i1.10916.
- HARB, Jean. Option-Critic Architecture: source code. GitHub. Disponível em: https://github.com/jeanharb/option_critic/tree/master/fourrooms.
- [Gymnasium. Cliff Walking](https://gymnasium.farama.org/v1.3.0/environments/toy_text/cliff_walking/)
- [Gymnasium. FrozenLake](https://gymnasium.farama.org/v1.3.0/environments/toy_text/frozen_lake/)
- [Gymnasium. Taxi](https://gymnasium.farama.org/v1.3.0/environments/toy_text/taxi/)
- SUTTON, Richard S.; BARTO, Andrew G. **Reinforcement Learning: An Introduction**. 2. ed. Cambridge: MIT Press, 2018.